# NL2SPARQL Test Notebook

## load libraries

In [32]:
from dotenv import load_dotenv
import os
import requests
import spacy
import time
from pathlib import Path
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    PromptTemplate
)
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib.plugins.sparql.parser import parseQuery
from rdflib.plugins.sparql.algebra import translateQuery
import json
import re

# Load environment variables from .env
load_dotenv()

True

## Basic LLM call to warhole server

In [5]:
# Get API URL and key from environment
OLLAMA_URL = os.getenv("OLLAMA_URL")
API_KEY = os.getenv("API_KEY")

if not API_KEY:
    raise ValueError("API_KEY not found. Please set it in your .env file.")
if not OLLAMA_URL:
    raise ValueError("OLLAMA_URL not found. Please set it in your .env file.")

payload = {
    "model": "llama3.3:70b",
    "messages": [
        {"role": "user", "content": "Hello?"}
    ]
}

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

response = requests.post(OLLAMA_URL, headers=headers, json=payload, stream=True)

for line in response.iter_lines():
    if line:
        print(line.decode("utf-8"))

{"id":"llama3.3:70b-87a2fa3b-c08e-4976-b8eb-a1b31f822d1b","created":1762697857,"model":"llama3.3:70b","choices":[{"index":0,"logprobs":null,"finish_reason":"stop","message":{"role":"assistant","content":"Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?"}}],"object":"chat.completion","usage":{"response_token/s":28.8,"prompt_token/s":88.39,"total_duration":12809071476,"load_duration":11803148494,"prompt_eval_count":12,"prompt_tokens":12,"prompt_eval_duration":135765914,"eval_count":25,"completion_tokens":25,"eval_duration":868130276,"approximate_total":"0h0m12s","total_tokens":37,"completion_tokens_details":{"reasoning_tokens":0,"accepted_prediction_tokens":0,"rejected_prediction_tokens":0}}}


## Mention Extraction

In [9]:
nlp = spacy.load("en_core_web_sm")

def extract_mentions(text: str):
    """
    Extracts entity mentions from the given text and returns a list
    of (entity_text, entity_label) pairs.
    """
    doc = nlp(text)
    mentions = [(ent.text, ent.label_) for ent in doc.ents]
    return mentions


## Identity Linking

In [10]:
# API endpoints based on DBLP documentation
DBLP_AUTHOR_API       = "https://dblp.org/search/author/api"
DBLP_VENUE_API        = "https://dblp.org/search/venue/api"
DBLP_PUBLICATION_API  = "https://dblp.org/search/publ/api"

def link_author_to_dblp(author_name: str, max_results: int = 5):
    params = {"q": author_name, "format": "json", "h": max_results}
    resp = requests.get(DBLP_AUTHOR_API, params=params)
    resp.raise_for_status()
    data = resp.json()
    hits = data.get("result", {}).get("hits", {}).get("hit", [])
    candidates = []
    for h in hits:
        info = h.get("info", {})
        candidates.append({
            "name":    info.get("author"),
            "url":     info.get("url"),
            "dblp_id": info.get("pid")
        })
    return candidates

def link_venue_to_dblp(venue_name: str, max_results: int = 5):
    params = {"q": venue_name, "format": "json", "h": max_results}
    resp = requests.get(DBLP_VENUE_API, params=params)
    resp.raise_for_status()
    data = resp.json()
    hits = data.get("result", {}).get("hits", {}).get("hit", [])
    candidates = []
    for h in hits:
        info = h.get("info", {})
        candidates.append({
            "name":    info.get("venue"),
            "url":     info.get("url"),
            "dblp_id": info.get("key")
        })
    return candidates

def link_publication_to_dblp(pub_title: str, max_results: int = 5):
    params = {"q": pub_title, "format": "json", "h": max_results}
    resp = requests.get(DBLP_PUBLICATION_API, params=params)
    resp.raise_for_status()
    data = resp.json()
    hits = data.get("result", {}).get("hits", {}).get("hit", [])
    candidates = []
    for h in hits:
        info = h.get("info", {})
        candidates.append({
            "title":   info.get("title"),
            "url":     info.get("url"),
            "dblp_id": info.get("key")
        })
    return candidates

def perform_linking(mentions):
    """
    For each (mention_text, label) pair:
      - if PERSON → link_author_to_dblp
      - if ORG (or maybe GPE) → treat as venue → link_venue_to_dblp
      - if WORK_OF_ART (or similar) → treat as publication → link_publication_to_dblp
    Returns list of dicts with mention, label and candidates (or skip note).
    """
    linked = []
    for text, label in mentions:
        entry = {"mention": text, "label": label, "candidates": None}
        try:
            if label == "PERSON":
                entry["candidates"] = link_author_to_dblp(text)
            elif label in ("ORG", "GPE"):
                entry["candidates"] = link_venue_to_dblp(text)
            elif label == "WORK_OF_ART":
                entry["candidates"] = link_publication_to_dblp(text)
            else:
                entry["note"] = "Skipping label {}".format(label)
        except Exception as e:
            entry["error"] = str(e)
        linked.append(entry)
        time.sleep(0.5)
    return linked

## Prompt Construction

In [11]:
def prompt_construction(mentions, linked_entities, question):
    # Read schema
    schema_path = Path("dblp_schema.rdf")
    with schema_path.open("r", encoding="utf-8") as f:
        dblp_schema = f.read()

    # load namespace cheat sheet
    with open("dblp_rdf_schema_cheatsheet.txt", "r", encoding="utf-8") as f:
        namespace_cheatsheet = f.read()

    examples = [
        {
            "question": "Return all papers published by Ian Goodfellow",
            "sparql": """PREFIX dblp: <https://dblp.org/rdf/schema#>
SELECT ?publication ?title WHERE {{
  VALUES ?author {{ <https://dblp.org/pid/43/7940> }}
  ?publication dblp:authoredBy ?author .
  ?publication dblp:title ?title .
}}"""
        },
        {
            "question": "Top 10 most frequent authors who have published at the International Semantic Web Conference (ISWC).",
            "sparql": """PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?name ?affiliation (COUNT(DISTINCT ?publ) as ?freq) (?pers as ?dblp) (SAMPLE(?orcids) as ?orcid)
WHERE {{
  ?stream <https://dblp.org/rdf/schema#primaryStreamTitle> "International Semantic Web Conference" .
  ?publ dblp:publishedInStream ?stream .
  ?publ dblp:authoredBy ?pers .
  ?pers rdfs:label ?name .
  OPTIONAL {{ ?pers dblp:primaryAffiliation ?affiliation . }}
  OPTIONAL {{ ?pers dblp:orcid ?orcids . }}
}}
GROUP BY ?name ?affiliation ?pers
ORDER BY DESC(?freq)
LIMIT 10"""
        },
        {
            "question": "Who are the highly cited coauthors of Andrej Karpathy?",
            "sparql": """PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX cito: <http://purl.org/spar/cito/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?name ?affiliation (COUNT(DISTINCT ?cite) AS ?cites) (?coauthor AS ?dblp) (SAMPLE(?orcids) AS ?orcid) WHERE {{
  VALUES ?author {{ <https://dblp.org/pid/04/9925> }} .
  ?copubl dblp:authoredBy ?author .
  ?copubl dblp:authoredBy ?coauthor .
  FILTER (?author != ?coauthor) .
  ?coauthor rdfs:label ?name .
  OPTIONAL {{ ?coauthor dblp:orcid ?orcids . }}
  OPTIONAL {{ ?coauthor dblp:primaryAffiliation ?affiliation . }}
  ?publ dblp:authoredBy ?coauthor .
  ?publ dblp:omid ?omid .
  ?cite cito:hasCitedEntity ?omid .
}}
GROUP BY ?name ?affiliation ?coauthor
ORDER BY DESC(?cites)
LIMIT 10"""
        }
    ]

    examples_text = "\n\n".join(
        f"Q: {ex['question']}\nSPARQL:\n{ex['sparql']}"
        for ex in examples
    )

    system_prompt_text = f"""
You are a SPARQL query generator for the dblp computer science bibliography RDF dataset.

Schema:
{dblp_schema}

Namespace Cheat Sheet:
{namespace_cheatsheet}

Guidelines:
- Make use of the prefixes provided in the schema (and explained in the namespace cheat sheet)
- Use the extracted and linked entities to inform your query construction
- Return **only** the SPARQL query, nothing else!
- Do **not** add any comments in the query

Examples:
{examples_text}
"""

    # System prompt (static)
    system_prompt = SystemMessagePromptTemplate(
        prompt=PromptTemplate(
            input_variables=[],
            template=system_prompt_text
        )
    )

    # Human prompt (dynamic)
    human_prompt = HumanMessagePromptTemplate(
        prompt=PromptTemplate(
            input_variables=["input_question", "extracted_entities", "linked_entities"],
            template="Question: {input_question}\nExtracted Entities: {extracted_entities}\nLinked Entities: {linked_entities}\n\nSPARQL query:"
        )
    )

    prompt_template = ChatPromptTemplate.from_messages([system_prompt, human_prompt])

    rendered_prompt = prompt_template.format(
        input_question=question,
        extracted_entities=mentions,
        linked_entities=linked_entities,
    )

    return rendered_prompt


## Query Generation via LLM Call

In [12]:
def query_llm(prompt: str) -> str:
    """
    Sends a prompt to the LLM and returns the response text.
    Prints token usage (prompt + completion).
    """

    # --- Read environment variables ---
    OLLAMA_URL = os.getenv("OLLAMA_URL")
    API_KEY = os.getenv("API_KEY")

    if not API_KEY:
        raise ValueError("API_KEY not found. Please set it in your .env file.")
    if not OLLAMA_URL:
        raise ValueError("OLLAMA_URL not found. Please set it in your .env file.")

    # --- Prepare the payload ---
    payload = {
        "model": "gpt-oss:20b",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    # --- Send the request (streaming) ---
    response = requests.post(OLLAMA_URL, headers=headers, json=payload, stream=True)

    # --- Parse the streamed response ---
    for line in response.iter_lines():
        if not line:
            continue

        data = json.loads(line.decode("utf-8"))
        answer = data["choices"][0]["message"]["content"].strip()

        usage = data.get("usage", {})
        prompt_tokens = usage.get("prompt_tokens")
        completion_tokens = usage.get("completion_tokens")

        print("\nNumber of input (prompt) tokens:", prompt_tokens)
        print("Number of output (completion) tokens:", completion_tokens)

        return answer

    return ""  # Edge case: no output received

## Query Validation

In [13]:
def validate_syntax(query: str):
    try:
        parseQuery(query)
        return True, "✅ SPARQL syntax is valid"
    except Exception as e:
        return False, f"❌ Syntax error: {e}"

## Run Query

In [14]:
def run_query(endpoint, query):
    s = SPARQLWrapper(endpoint)
    s.setMethod("POST")
    s.setTimeout(60)
    s.setReturnFormat(JSON)
    s.addParameter("format", "json")
    s.addCustomHttpHeader("User-Agent", "NL2SPARQL/0.1 (julius.kaltwasser@rwth-aachen.de)")
    s.setQuery(query)
    return s.query().convert()["results"]["bindings"]

## User Question

In [ ]:
question = "Who are the highly cited coauthors of Hannah Bast?"
question2 = "Who are the authors of the paper named: Attentions is all you need?"
question3 = "Return the 10 most cited papers of Nicholas Carlini"
question4 = "Database papers published in the Semantic Web Journal."
question5  = "Which papers did Ian Goodfellow publish after 2018?"
question6  = "What are the most cited papers on federated learning?"
question7  = "List all coauthors of Yoshua Bengio."
question8  = "How many papers has Andrew Ng published in NeurIPS?"
question9  = "Show papers that cite “Attention Is All You Need.”"
question10 = "What are the top 5 most cited papers in computer vision?"
question11 = "Who collaborated with Geoffrey Hinton on deep learning papers?"
question12 = "Which authors have published in both CVPR and ICCV?"
question13 = "List all papers with 'graph neural networks' in the title."
question14 = "Which conferences did the paper 'BERT: Pre-training of Deep Bidirectional Transformers' appear in?"
question15 = "Who are the most cited researchers in natural language processing?"
question16 = "Return papers coauthored by Jürgen Schmidhuber and Sepp Hochreiter."
question17 = "Find the average citation count of papers published in ICML 2020."
question18 = "Which journals have published papers about 'adversarial examples'?"
question19 = "Show all papers published by OpenAI authors."
question20 = "Which paper by Yann LeCun has the highest number of citations?"
question21 = "Who are the editors of the journal 'Machine Learning'?"
question22 = "List all papers that reference 'Generative Adversarial Networks.'"
question23 = "Which institutions are most active in quantum computing research?"
question24 = "Find all papers authored by people affiliated with Stanford University in 2023."


## Test

In [ ]:
mentions_ = extract_mentions(question)
print("Extracted Mentions:", mentions_)
linking_results_ = perform_linking(mentions_)
print("Linking Results:", linking_results_)
rendered_prompt_ = prompt_construction(mentions_, linking_results_, question)
#print(rendered_prompt)
query_ = query_llm(rendered_prompt_)
print("Generated SPARQL Query:\n", query_)
validation_result_, msg_ = validate_syntax(query_)
print("Syntax Validation Result:", msg_)
if validation_result_:
    endpoint = "https://sparql.dblp.org/sparql"
    results_ = run_query(endpoint, query_)
    print("Query Results:", results_)

Extracted Mentions: [('Hannah Bast', 'PERSON')]
Linking Results: [{'mention': 'Hannah Bast', 'label': 'PERSON', 'candidates': [{'name': 'Hannah Bast', 'url': 'https://dblp.org/pid/b/HannahBast', 'dblp_id': None}]}]

Number of input (prompt) tokens: 8192
Number of output (completion) tokens: 632
Generated SPARQL Query:
 PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX cito: <http://purl.org/spar/cito/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?name ?affiliation (COUNT(DISTINCT ?cite) AS ?cites) (?coauthor AS ?dblp) (SAMPLE(?orcids) AS ?orcid) WHERE {
  VALUES ?author { <https://dblp.org/pid/b/HannahBast> } .
  ?copubl dblp:authoredBy ?author .
  ?copubl dblp:authoredBy ?coauthor .
  FILTER (?author != ?coauthor) .
  ?coauthor rdfs:label ?name .
  OPTIONAL { ?coauthor dblp:orcid ?orcids . }
  OPTIONAL { ?coauthor dblp:primaryAffiliation ?affiliation . }
  ?publ dblp:authoredBy ?coauthor .
  ?publ dblp:omid ?omid .
  ?cite cito:hasCitedEntity ?omid .
}
GROUP BY ?name 

## SPARQL-Comparison

In [ ]:
def canonicalize_sparql(query_str):
    """
    Parse and canonicalize a SPARQL query:
    - Parse into abstract syntax tree
    - Extract WHERE triples and filters
    - Sort them for stable comparison
    """
    try:
        # Parse the query
        parsed = parseQuery(query_str)
        algebra = translateQuery(parsed)

        # Get WHERE clause (triples)
        triples = []
        filters = []

        def extract_patterns(expr):
            # Recursively extract triples and filters
            if hasattr(expr, 'part'):
                extract_patterns(expr.part)
            elif hasattr(expr, 'p'):
                extract_patterns(expr.p)
            elif hasattr(expr, 'triples'):
                for t in expr.triples:
                    triples.append(tuple(map(str, t)))
            if hasattr(expr, 'expr'):
                extract_patterns(expr.expr)
            if hasattr(expr, 'p1'):
                extract_patterns(expr.p1)
            if hasattr(expr, 'p2'):
                extract_patterns(expr.p2)
            if hasattr(expr, 'p3'):
                extract_patterns(expr.p3)
            if hasattr(expr, 'expr'):
                filters.append(str(expr.expr))
            if hasattr(expr, 'graph'):
                extract_patterns(expr.graph)
            if hasattr(expr, 'name'):
                extract_patterns(expr.name)

        extract_patterns(algebra.algebra)

        # Sort for canonical form
        triples_sorted = sorted(set(triples))
        filters_sorted = sorted(set(filters))

        return {
            "triples": triples_sorted,
            "filters": filters_sorted
        }

    except Exception as e:
        return {"error": str(e)}
    
def compare_queries(q1, q2):
    c1 = canonicalize_sparql(q1)
    c2 = canonicalize_sparql(q2)
    if "error" in c1 or "error" in c2:
        return False, (c1, c2)
    return c1 == c2, (c1, c2)

#testing this Functions:
'''
# Query 1
query1 = "PREFIX foaf: <http://xmlns.com/foaf/0.1/> PREFIX dbo: <http://dbpedia.org/ontology/> SELECT ?person ?name ?birthPlace WHERE { ?person a foaf:Person . ?person foaf:name ?name . ?person dbo:birthPlace ?birthPlace . }"

# Query 2 (Reihenfolge vertauscht, Semikolon verwendet)
query2 = "PREFIX foaf: <http://xmlns.com/foaf/0.1/> PREFIX dbo: <http://dbpedia.org/ontology/> SELECT ?person ?name ?birthPlace WHERE { ?person a foaf:Person ; dbo:birthPlace ?birthPlace ; foaf:name ?name . }"


match, details = compare_queries(query1, query2)

if match:
    print("✅ Query matches the ground truth (canonical form).")
else:
    print("❌ Query differs from ground truth.")
    print("Generated canonical form:", details[0])
    print("Ground truth canonical form:", details[1])
    '''

✅ Query matches the ground truth (canonical form).


## Test Loop

enter Tests with this Template:

In [ ]:
"""

"QUESTION_HERE": {
        "Entities": {
            "": {
                "label": "",
                "uri": ""
            }
        },
        "Query": {
            "raw": "", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    }


    
#Example for test entry:
"Who are the highly cited coauthors of Hannah Bast?": {
        "Entities": {
            "person": {
                "label": "Hannah Bast",
                "uri": "https://dblp.org/pid/b/HannahBast"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX cito: [http://purl.org/spar/cito/](http://purl.org/spar/cito/) PREFIX rdfs: [http://www.w3.org/2000/01/rdf-schema#](http://www.w3.org/2000/01/rdf-schema#) SELECT ?name ?affiliation (COUNT(DISTINCT ?cite) as ?cites) (?coauthor as ?dblp) (SAMPLE(?orcids) as ?orcid) WHERE { VALUES ?author { [https://dblp.org/pid/b/HannahBast](https://dblp.org/pid/b/HannahBast) } . ?copubl dblp:authoredBy ?author . ?copubl dblp:authoredBy ?coauthor . FILTER ( ?author != ?coauthor ) . ?coauthor rdfs:label ?name . OPTIONAL { ?coauthor dblp:orcid ?orcids . } OPTIONAL { ?coauthor dblp:primaryAffiliation ?affiliation . } ?publ dblp:authoredBy ?coauthor . ?publ dblp:omid ?omid . ?cite cito:hasCitedEntity ?omid . } GROUP BY ?name ?affiliation ?coauthor ORDER BY DESC(?cites) LIMIT 10",
            "triples": [#example not used but could be..?
                ["Could be", "added", "Like This"],
                ["?publication", "dblp:authoredBy", "<https://dblp.org/pid/43/7940>"],
                ["?publication", "dblp:title", "?title"],
                ["?publication", "dblp:yearOfPublication", "?year"]
            ],
            "filters": ["?year > 2018"]#example not used but could be..?
        }
    }
"""

'\n\n"QUESTION_HERE": {\n        "Entities": {\n            "": {\n                "label": "",\n                "uri": ""\n            }\n        },\n        "Query": {\n            "raw": "", #Query in one line\n            "triples": []#not used but could be..?,\n            "filters": []#not used but could be..?\n        }\n    }\n\n\n\n#Example for test entry:\n"Who are the highly cited coauthors of Hannah Bast?": {\n        "Entities": {\n            "person": {\n                "label": "Hannah Bast",\n                "uri": "https://dblp.org/pid/b/HannahBast"\n            }\n        },\n        "Query": {\n            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX cito: [http://purl.org/spar/cito/](http://purl.org/spar/cito/) PREFIX rdfs: [http://www.w3.org/2000/01/rdf-schema#](http://www.w3.org/2000/01/rdf-schema#) SELECT ?name ?affiliation (COUNT(DISTINCT ?cite) as ?cites) (?coauthor as ?dblp) (SAMPLE(?orcids) as ?orcid) WHERE { VALUE

In [27]:
tests = {
    "Who are the highly cited coauthors of Hannah Bast?": {
        "Entities": {
            "person": {
                "label": "Hannah Bast",
                "uri": "https://dblp.org/pid/b/HannahBast"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX cito: [http://purl.org/spar/cito/](http://purl.org/spar/cito/) PREFIX rdfs: [http://www.w3.org/2000/01/rdf-schema#](http://www.w3.org/2000/01/rdf-schema#) SELECT ?name ?affiliation (COUNT(DISTINCT ?cite) as ?cites) (?coauthor as ?dblp) (SAMPLE(?orcids) as ?orcid) WHERE { VALUES ?author { [https://dblp.org/pid/b/HannahBast](https://dblp.org/pid/b/HannahBast) } . ?copubl dblp:authoredBy ?author . ?copubl dblp:authoredBy ?coauthor . FILTER ( ?author != ?coauthor ) . ?coauthor rdfs:label ?name . OPTIONAL { ?coauthor dblp:orcid ?orcids . } OPTIONAL { ?coauthor dblp:primaryAffiliation ?affiliation . } ?publ dblp:authoredBy ?coauthor . ?publ dblp:omid ?omid . ?cite cito:hasCitedEntity ?omid . } GROUP BY ?name ?affiliation ?coauthor ORDER BY DESC(?cites) LIMIT 10",
            "triples": [#example not used but could be..?
                ["Could be", "added", "Like This"],
                ["?publication", "dblp:authoredBy", "<https://dblp.org/pid/43/7940>"],
                ["?publication", "dblp:title", "?title"],
                ["?publication", "dblp:yearOfPublication", "?year"]
            ],
            "filters": ["?year > 2018"]#example not used but could be..?
        }
    },
    "Who are the authors of the paper named: Attentions is all you need?": {
        "Entities": {
            "publication": {
                "label": "Ashish Vaswani et al.: Attention is All you Need. (2017)",
                "uri": "https://dblp.org/rec/conf/nips/VaswaniSPUJGKP17"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?author ?author_name WHERE { VALUES ?publication { [https://dblp.org/rec/conf/nips/VaswaniSPUJGKP17](https://dblp.org/rec/conf/nips/VaswaniSPUJGKP17) } . ?publication dblp:authoredBy ?author . ?author dblp:creatorName ?author_name . }",
            "triples": [#example not used but could be..?
                ["Could be", "added", "Like This"],
                ["?publication", "dblp:authoredBy", "<https://dblp.org/pid/43/7940>"],
                ["?publication", "dblp:title", "?title"],
                ["?publication", "dblp:yearOfPublication", "?year"]
            ],
            "filters": ["?year > 2018"]#example not used but could be..?
        }
    },
    "Return the 10 most cited papers of Nicholas Carlini": {
        "Entities": {
            "person": {
                "label": "Nicholas Carlini",
                "uri": "https://dblp.org/pid/145/1806"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX cito: [http://purl.org/spar/cito/](http://purl.org/spar/cito/) SELECT ?paper ?title (COUNT(?cites) AS ?citation_count) WHERE { ?paper dblp:authoredBy [https://dblp.org/pid/145/1806](https://dblp.org/pid/145/1806) . ?paper dblp:title ?title . ?paper dblp:omid ?omid . ?cite cito:hasCitedEntity ?omid . } GROUP BY ?paper ?title ORDER BY DESC(?citation_count) LIMIT 10",
            "triples": [#example not used but could be..?
                ["Could be", "added", "Like This"],
                ["?publication", "dblp:authoredBy", "<https://dblp.org/pid/43/7940>"],
                ["?publication", "dblp:title", "?title"],
                ["?publication", "dblp:yearOfPublication", "?year"]
            ],
            "filters": ["?year > 2018"]#example not used but could be..?
        }
    },
    "Database papers published in the Semantic Web Journal.": {
        "Entities": {
            "publication": {
                "label": "Keith G. Jeffery: Database Conference Calendar / Calls For Papers. (1993)",
                "uri": "https://dblp.org/rec/journals/sigmod/Jeffery93"
            },
            "venue": {
                "label": "Journal of Web Semantics",
                "uri": "https://dblp.org/streams/journals/ws"
            }
        },
        "Query": {
            "raw": """PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?paper ?title WHERE { ?paper dblp:publishedInStream [https://dblp.org/streams/journals/ws](https://dblp.org/streams/journals/ws) . ?paper dblp:bibtexType "article" . ?paper dblp:title ?title . }""",
            "triples": [#example not used but could be..?
                ["Could be", "added", "Like This"],
                ["?publication", "dblp:authoredBy", "<https://dblp.org/pid/43/7940>"],
                ["?publication", "dblp:title", "?title"],
                ["?publication", "dblp:yearOfPublication", "?year"]
            ],
            "filters": ["?year > 2018"]#example not used but could be..?
        }
    },
    "Which papers did Ian Goodfellow publish after 2018?": {
        "Entities": {
            "person": {
                "label": "Ian J. Goodfellow",
                "uri": "https://dblp.org/pid/43/7940"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?title ?year WHERE { VALUES ?author { [https://dblp.org/pid/43/7940](https://dblp.org/pid/43/7940) } ?publication dblp:authoredBy ?author . ?publication dblp:title ?title . ?publication dblp:yearOfPublication ?year . FILTER (?year > 2018) }",
            "triples": [#example not used but could be..?
                ["Could be", "added", "Like This"],
                ["?publication", "dblp:authoredBy", "<https://dblp.org/pid/43/7940>"],
                ["?publication", "dblp:title", "?title"],
                ["?publication", "dblp:yearOfPublication", "?year"]
            ],
            "filters": ["?year > 2018"]#example not used but could be..?
        }
    },
    "What are the most cited papers on federated learning?": {
        "Entities": {
            "publication": {
                "label": "Tianli Gao et al.: Federated Learning Meets Network Coding: Efficient Coded Hierarchical Federated Learning. (2024)",
                "uri": "https://dblp.org/rec/conf/itw/GaoLLTG24"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX cito: [http://purl.org/spar/cito/](http://purl.org/spar/cito/) SELECT ?title ?cites WHERE { ?publ dblp:title ?title ; dblp:omid ?omid . ?cite cito:hasCitedEntity ?omid . } GROUP BY ?title ?cites ORDER BY DESC(?cites) LIMIT 10",
            "triples": [#example not used but could be..?
                ["Could be", "added", "Like This"],
                ["?publication", "dblp:authoredBy", "<https://dblp.org/pid/43/7940>"],
                ["?publication", "dblp:title", "?title"],
                ["?publication", "dblp:yearOfPublication", "?year"]
            ],
            "filters": ["?year > 2018"]#example not used but could be..?
        }
    },
    "List all coauthors of Yoshua Bengio.": {
        "Entities": {
            "person": {
                "label": "Yoshua Bengio",
                "uri": "https://dblp.org/pid/56/953"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?coauthor ?coauthor_name WHERE { VALUES ?author { [https://dblp.org/pid/56/953](https://dblp.org/pid/56/953) } . ?publication dblp:authoredBy ?author . ?publication dblp:authoredBy ?coauthor . FILTER (?author != ?coauthor) ?coauthor dblp:creatorName ?coauthor_name . }", 
            "triples": [],#not used but could be..?,
            "filters": []#not used but could be..?
        }
    },
    "How many papers has Andrew Ng published in NeurIPS?": {
        "Entities": {
            "person": {
                "label": "Andrew Y. Ng",
                "uri": "https://dblp.org/pid/n/AndrewYNg"
            },
            "publication": {
                "label": "Regular Papers. (2020)",
                "uri": "https://dblp.org/rec/journals/ia/X20a"
            },
            "venue": {
                "label": "Conference on Neural Information Processing Systems (NeurIPS)",
                "uri": "https://dblp.org/streams/conf/nips"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT (COUNT(?paper) AS ?numberOfPapers) WHERE { VALUES ?author { [https://dblp.org/pid/n/AndrewYNg](https://dblp.org/pid/n/AndrewYNg) } VALUES ?venue { [https://dblp.org/streams/conf/nips](https://dblp.org/streams/conf/nips) } ?paper dblp:authoredBy ?author . ?paper dblp:publishedInStream ?venue . }", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Show papers that cite “Attention Is All You Need.”": {
        "Entities": {
            "publication": {
                "label": "Cem Subakan et al.: Attention Is All You Need In Speech Separation. (2021)",
                "uri": "https://dblp.org/rec/conf/icassp/SubakanRCBZ21"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX cito: [http://purl.org/spar/cito/](http://purl.org/spar/cito/) SELECT ?citing_paper ?citing_paper_title WHERE { VALUES ?cited_paper { [https://dblp.org/rec/conf/icassp/SubakanRCBZ21](https://dblp.org/rec/conf/icassp/SubakanRCBZ21) } ?citing_paper cito:cites ?cited_paper . ?citing_paper dblp:title ?citing_paper_title . }", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "What are the top 5 most cited papers in computer vision?": {
        "Entities": {
            "publication": {
                "label": "Most Cited Paper Award. (2008)",
                "uri": "https://dblp.org/rec/journals/iwc/X08"
            },
            "venue": {
                "label": "International Conference on Computer Vision (VISION)",
                "uri": "https://dblp.org/streams/conf/vision"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX rdfs: [http://www.w3.org/2000/01/rdf-schema#](http://www.w3.org/2000/01/rdf-schema#) SELECT ?paper ?title WHERE { ?paper dblp:publishedInStream [https://dblp.org/streams/conf/vision](https://dblp.org/streams/conf/vision) . ?paper dblp:title ?title . } LIMIT 5", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Who collaborated with Geoffrey Hinton on deep learning papers?": {
        "Entities": {
            "person": {
                "label": "Geoffrey E. Hinton",
                "uri": "https://dblp.org/pid/10/3248"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?coauthorName ?coauthorAffiliation WHERE { VALUES ?author { [https://dblp.org/pid/10/3248](https://dblp.org/pid/10/3248) } . ?paper dblp:authoredBy ?author . ?paper dblp:title ?title . FILTER(CONTAINS(UCASE(str(?title)), 'DEEP LEARNING')) ?paper dblp:authoredBy ?coauthor . FILTER(?coauthor != ?author) ?coauthor dblp:creatorName ?coauthorName . OPTIONAL { ?coauthor dblp:primaryAffiliation ?coauthorAffiliation . } } GROUP BY ?coauthorName ?coauthorAffiliation", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Which authors have published in both CVPR and ICCV?": {
        "Entities": {
            "venue": {
                "label": "IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)",
                "uri": "https://dblp.org/streams/conf/cvpr"
            },
            "venue": {
                "label": "IEEE International Conference on Computer Vision (ICCV)",
                "uri": "https://dblp.org/streams/conf/iccv"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?author ?author_name WHERE { ?pub1 dblp:publishedInStream [https://dblp.org/streams/conf/cvpr](https://dblp.org/streams/conf/cvpr) . ?pub1 dblp:authoredBy ?author . ?pub2 dblp:publishedInStream [https://dblp.org/streams/conf/iccv](https://dblp.org/streams/conf/iccv) . ?pub2 dblp:authoredBy ?author . ?author dblp:creatorName ?author_name . }", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "List all papers with 'graph neural networks' in the title.": {
        "Entities": {
            "publication": {
                "label": "Yupeng Hou et al.: Neural Graph Matching for Pre-training Graph Neural Networks. (2022)",
                "uri": "https://dblp.org/rec/conf/sdm/HouHZ00W22"
            }
        },
        "Query": {
            "raw": """PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?publication ?title WHERE { ?publication dblp:title ?title . FILTER regex(?title, "graph neural networks", "i") . }""", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Which conferences did the paper 'BERT: Pre-training of Deep Bidirectional Transformers' appear in?": {
        "Entities": {
            "publication": {
                "label": "Jacob Devlin et al.: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. (2019)",
                "uri": "https://dblp.org/rec/conf/naacl/DevlinCLT19"
            },
            "venue": {
                "label": "\"I Can't Believe It's Not Better!\" Workshop Series (ICBINB)",
                "uri": "https://dblp.org/streams/conf/icbinb"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?venue ?venue_name WHERE { VALUES ?publication { [https://dblp.org/rec/conf/naacl/DevlinCLT19](https://dblp.org/rec/conf/naacl/DevlinCLT19) } ?publication dblp:publishedInStream ?venue . ?venue dblp:streamTitle ?venue_name . }", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Who are the most cited researchers in natural language processing?": {
        "Entities": {
            "person": {
                "label": "Most. Mahjabin",
                "uri": "https://dblp.org/pid/326/6846"
            }
        },
        "Query": {
            "raw": """PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX cito: [http://purl.org/spar/cito/](http://purl.org/spar/cito/) PREFIX rdfs: [http://www.w3.org/2000/01/rdf-schema#](http://www.w3.org/2000/01/rdf-schema#) SELECT ?name ?affiliation (COUNT(DISTINCT ?cite) AS ?cites) WHERE { ?publ dblp:authoredBy ?creator . ?creator dblp:primaryAffiliation ?affiliation . FILTER CONTAINS(?affiliation, "Natural language processing") ?creator rdfs:label ?name . ?publ dblp:omid ?omid . ?cite cito:hasCitedEntity ?omid . } GROUP BY ?name ?affiliation ORDER BY DESC(?cites) LIMIT 10""", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Return papers coauthored by Jürgen Schmidhuber and Sepp Hochreiter.": {
        "Entities": {
            "person": {
                "label": "J\u00FCrgen Schmidhuber",
                "uri": "https://dblp.org/pid/s/JurgenSchmidhuber"
            },
            "person": {
                "label": "Sepp Hochreiter",
                "uri": "https://dblp.org/pid/h/SeppHochreiter"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?publication ?title ?coauthor WHERE { VALUES ?author1 { [https://dblp.org/pid/s/JurgenSchmidhuber](https://dblp.org/pid/s/JurgenSchmidhuber) } . VALUES ?author2 { [https://dblp.org/pid/h/SeppHochreiter](https://dblp.org/pid/h/SeppHochreiter) } . ?publication dblp:authoredBy ?author1 . ?publication dblp:authoredBy ?author2 . ?publication dblp:title ?title . ?publication dblp:authoredBy ?coauthor . FILTER(?coauthor != ?author1 && ?coauthor != ?author2) }", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Find the average citation count of papers published in ICML 2020.": {
        "Entities": {
            "venue": {
                "label": "International Conference on Machine Learning (ICML)",
                "uri": "https://dblp.org/streams/conf/icml"
            },
            "publication": {
                "label": "Andreas Holzinger et al.: xxAI - Beyond Explainable AI - International Workshop, Held in Conjunction with ICML 2020, July 18, 2020, Vienna, Austria, Revised and Extended Papers (2022)",
                "uri": "https://dblp.org/rec/conf/icml/2020xxai"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX cito: [http://purl.org/spar/cito/](http://purl.org/spar/cito/) SELECT (AVG(?cites) AS ?averageCitationCount) WHERE { ?publ dblp:publishedInStream [https://dblp.org/streams/conf/icml](https://dblp.org/streams/conf/icml) . ?publ dblp:yearOfPublication 2020 . ?publ dblp:omid ?omid . ?cite cito:hasCitedEntity ?omid . } GROUP BY ?publ", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Which journals have published papers about 'adversarial examples'?": {
        "Entities": {
            "publication": {
                "label": "Chang Xiao and Changxi Zheng: One Man's Trash Is Another Man's Treasure: Resisting Adversarial Examples by Adversarial Examples. (2020)",
                "uri": "https://dblp.org/rec/conf/cvpr/XiaoZ20"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT DISTINCT ?journal WHERE { VALUES ?publication { [https://dblp.org/rec/conf/cvpr/XiaoZ20](https://dblp.org/rec/conf/cvpr/XiaoZ20) } . ?publication dblp:publishedInJournal ?journal . }", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Show all papers published by OpenAI authors.": {
        "Entities": {
            "publication": {
                "label": "Regular Papers. (2017)",
                "uri": "https://dblp.org/rec/journals/semweb/X17a"
            },
            "person": {
                "label": "OpenAI",
                "uri": "https://dblp.org/pid/225/4716"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?paper ?title WHERE { VALUES ?author { [https://dblp.org/pid/225/4716](https://dblp.org/pid/225/4716) } . ?paper dblp:authoredBy ?author . ?paper dblp:title ?title . }", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Which paper by Yann LeCun has the highest number of citations?": {
        "Entities": {
            "person": {
                "label": "Yann LeCun",
                "uri": "https://dblp.org/pid/l/YannLeCun"
            },
            "publication": {
                "label": "Yann LeCun et al.: Effiicient BackProp. (1996)",
                "uri": "https://dblp.org/rec/conf/nips/LeCunBOM96"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX cito: [http://purl.org/spar/cito/](http://purl.org/spar/cito/) SELECT ?paper ?title (COUNT(DISTINCT ?citation) as ?citations) WHERE { ?paper dblp:authoredBy [https://dblp.org/pid/l/YannLeCun](https://dblp.org/pid/l/YannLeCun) . ?paper dblp:title ?title . ?paper dblp:omid ?omid . ?citation cito:hasCitedEntity ?omid . } GROUP BY ?paper ?title ORDER BY DESC(?citations) LIMIT 1", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Who are the editors of the journal 'Machine Learning'?": {
        "Entities": {
            "publication": {
                "label": "Machine Learning. (2018)",
                "uri": "https://dblp.org/rec/reference/snam/X18ta"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?editor ?editorName WHERE { VALUES ?publication { [https://dblp.org/rec/reference/snam/X18ta](https://dblp.org/rec/reference/snam/X18ta) } ?publication dblp:editedBy ?editor . ?editor dblp:creatorname ?editorName . }", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "List all papers that reference 'Generative Adversarial Networks.'": {
        "Entities": {
            "publication": {
                "label": "Moez Krichen: Generative Adversarial Networks. (2023)",
                "uri": "https://dblp.org/rec/conf/icccnt/Krichen23"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX cito: [http://purl.org/spar/cito/](http://purl.org/spar/cito/) SELECT ?paper ?title WHERE { ?paper cito:hasCitedEntity [https://dblp.org/rec/conf/icccnt/Krichen23](https://dblp.org/rec/conf/icccnt/Krichen23) . ?paper dblp:title ?title . }", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Which institutions are most active in quantum computing research?": {
        "Entities": {
            "venue": {
                "label": "Quantum Computing and Quantum Communications (QCQC)",
                "uri": "https://dblp.org/streams/conf/qcqc"
            }
        },
        "Query": {
            "raw": "PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) PREFIX rdfs: [http://www.w3.org/2000/01/rdf-schema#](http://www.w3.org/2000/01/rdf-schema#) SELECT ?institution (COUNT(DISTINCT ?publ) as ?activity) WHERE { VALUES ?venue { [https://dblp.org/streams/conf/qcqc](https://dblp.org/streams/conf/qcqc) } . ?publ dblp:publishedInStream ?venue . ?publ dblp:authoredBy ?creator . ?creator dblp:affiliation ?institution . } GROUP BY ?institution ORDER BY DESC(?activity) LIMIT 10", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    },
    "Find all papers authored by people affiliated with Stanford University in 2023.": {
        "Entities": {
        },
        "Query": {
            "raw": """PREFIX dblp: [https://dblp.org/rdf/schema#](https://dblp.org/rdf/schema#) SELECT ?paper ?title WHERE { ?paper dblp:authoredBy ?author . ?author dblp:affiliation "Stanford University" . ?paper dblp:yearOfPublication 2023 . ?paper dblp:title ?title . }""", #Query in one line
            "triples": [],#not used but could be..?
            "filters": []#not used but could be..?
        }
    }
}


## The Pipeline

In [30]:
def run_pipeline(question):
    print("our Pipeline results:")
    mentions = extract_mentions(question)
    print("Extracted Mentions:", mentions)
    linking_results = perform_linking(mentions)
    print("Linking Results:", linking_results)
    rendered_prompt = prompt_construction(mentions, linking_results, question)
    #print(rendered_prompt)
    query = query_llm(rendered_prompt)
    print("Generated SPARQL Query:\n", query)
    validation_result, msg = validate_syntax(query)
    print("Syntax Validation Result:", msg)
    if validation_result:
        endpoint = "https://sparql.dblp.org/sparql"
        results = run_query(endpoint, query)
        print("Query Results:", results)
    return (linking_results, query)

## Similarity Score of two Querys

In [ ]:
def query_similarity(gt_query, generated_query):
    """
    gt_query & generated_query: dicts mit keys: prefixes, select, where, filters
    returns a score in [0,1]
    """

    # PREFIXES
    gt_prefixes = set(gt_query.get("prefixes", []))
    gen_prefixes = set(generated_query.get("prefixes", []))
    prefix_score = len(gt_prefixes & gen_prefixes) / max(len(gt_prefixes), 1)

    # SELECT
    gt_select = set(gt_query.get("select", []))
    gen_select = set(generated_query.get("select", []))
    select_score = len(gt_select & gen_select) / max(len(gt_select), 1)

    # WHERE / Triples
    gt_triples = set(tuple(sorted(t.values())) for t in gt_query.get("where", []))
    gen_triples = set(tuple(sorted(t.values())) for t in generated_query.get("where", []))
    triple_score = len(gt_triples & gen_triples) / max(len(gt_triples), 1)

    # FILTERS
    gt_filters = set(gt_query.get("filters", []))
    gen_filters = set(generated_query.get("filters", []))
    filter_score = len(gt_filters & gen_filters) / max(len(gt_filters), 1)

    # Weighted score
    score = 0.2*prefix_score + 0.2*select_score + 0.5*triple_score + 0.1*filter_score

    return score

def query_to_struct(query_str):
    """
    Convert a SPARQL query string to a structured dict:
    {
        'prefixes': [...],
        'select': [...],
        'triples': [...],
        'filters': [...]
    }
    """

    # --- extract Prefixes ---
    prefixes = re.findall(r'PREFIX\s+[\w]*:\s*<[^>]+>', query_str)

    # --- extract SELECT variables ---
    select_match = re.search(r'SELECT\s+(DISTINCT\s+)?(.*?)\s+WHERE', query_str, re.IGNORECASE | re.DOTALL)
    if select_match:
        select_vars = select_match.group(2).split()
    else:
        select_vars = []

    # --- extract Triples & Filter via canonicalize_sparql ---
    canon = canonicalize_sparql(query_str)
    triples = []
    for t in canon.get("triples", []):
        if len(t) == 3:
            triples.append({"subject": t[0], "predicate": t[1], "object": t[2]})
    filters = canon.get("filters", [])

    return {
        "prefixes": prefixes,
        "select": select_vars,
        "where": triples,
        "filters": filters
    }

## The Loop

In [ ]:
correct = 0
similarity_score = 0
for question, info in tests.items():
    print(f"Frage: {question}")

    #run our pipeline:
    try:
        (linking_results, query) = run_pipeline(question)
    except Exception as e:
        print(f"An error occurred in the Pipeline with Question: {question}")
        continue
    
    # Mention & Entity-linking test
    print("\n Entity linking check:")
    gt_entities = info["Entities"]
    print("Expected Entities:", gt_entities)
    print("Predicted Entities:", linking_results)


    # SPARQL Query comparison
    print("\n SPARQL Query comparison:")
    match, details = compare_queries(query,info["Query"]["raw"])
    struct1 = query_to_struct(query)
    struct2 = query_to_struct(info["Query"]["raw"])

    # Calculate accuracy score
    score = query_similarity(struct1, struct2)
    similarity_score += score
    print(f"Query similarity: {score*100:.1f}%")

    if match:
        correct += 1
        print("✅ Query matches the ground truth (canonical form).")
    else:
        print("❌ Query differs from ground truth.")
        print("Generated canonical form:", details[0])
        print("Ground truth canonical form:", details[1])

print(f"{correct} out of {len(tests)} tests successful ({correct/len(tests)*100:.1f}%)")
print(f"Query similarity overall average: {(similarity_score/len(tests))*100:.1f}%")


    # Results comparison (Paper finds)

Frage: Who are the highly cited coauthors of Hannah Bast?
our Pipeline results:
Extracted Mentions: [('Hannah Bast', 'PERSON')]
Linking Results: [{'mention': 'Hannah Bast', 'label': 'PERSON', 'candidates': [{'name': 'Hannah Bast', 'url': 'https://dblp.org/pid/b/HannahBast', 'dblp_id': None}]}]

Number of input (prompt) tokens: 8192
Number of output (completion) tokens: 1015
Generated SPARQL Query:
 PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX cito: <http://purl.org/spar/cito/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?name ?affiliation (COUNT(DISTINCT ?cite) AS ?cites) (?coauthor AS ?dblp) (SAMPLE(?orcids) AS ?orcid) WHERE {
  VALUES ?author { <https://dblp.org/pid/b/HannahBast> } .
  ?copubl dblp:authoredBy ?author .
  ?copubl dblp:authoredBy ?coauthor .
  FILTER (?author != ?coauthor) .
  ?coauthor rdfs:label ?name .
  OPTIONAL { ?coauthor dblp:orcid ?orcids . }
  OPTIONAL { ?coauthor dblp:primaryAffiliation ?affiliation . }
  ?coauthor_pub dblp:authoredBy ?c